<a href="https://colab.research.google.com/github/barney-rai/asl-recognition/blob/main/asl_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
from google.colab import drive

drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [49]:
import os
import cv2
import numpy as np
import torch

from torch.utils.data import TensorDataset, DataLoader

In [50]:
A_DIR = "/content/drive/MyDrive/ASL_Project/A"
B_DIR = "/content/drive/MyDrive/ASL_Project/B"

print("A images:", len(os.listdir(A_DIR)))
print("B images:", len(os.listdir(B_DIR)))

A images: 3000
B images: 3000


In [51]:
print(torch.__version__)

2.11.0+cu128


In [52]:
print(torch.cuda.is_available())

True


In [53]:
print(torch.cuda.get_device_name(0))

Tesla T4


In [54]:
#use gpu is possible. else use cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [55]:
images = []
labels = []

In [56]:
for filename in os.listdir(A_DIR):
    image_path = os.path.join(A_DIR, filename)

    img = cv2.imread(image_path)

    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (64, 64))

    images.append(resized)
    labels.append(0)

In [57]:
for filename in os.listdir(B_DIR):
    image_path = os.path.join(B_DIR, filename)

    img = cv2.imread(image_path)

    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (64, 64))

    images.append(resized)
    labels.append(1)

In [58]:
X = np.array(images)
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (6000, 64, 64)
y shape: (6000,)


In [59]:
X = X / 255.0

print("Minimum:", X.min())
print("Maximum:", X.max())

Minimum: 0.0
Maximum: 1.0


In [60]:
print(type(X))
print(type(y))

print(X.shape)
print(y.shape)

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
(6000, 64, 64)
(6000,)


In [61]:
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

In [62]:
print("X:", X_tensor.shape)
print("y:", y_tensor.shape)

print("X dtype:", X_tensor.dtype)
print("y dtype:", y_tensor.dtype)

X: torch.Size([6000, 64, 64])
y: torch.Size([6000])
X dtype: torch.float32
y dtype: torch.int64


In [63]:
dataset = TensorDataset(X_tensor, y_tensor)
print("Number of examples:", len(dataset))

Number of examples: 6000


In [64]:
image, label = dataset[0]

print("Image shape:", image.shape)
print("Label:", label)

Image shape: torch.Size([64, 64])
Label: tensor(0)


In [65]:
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

In [66]:
images_batch, labels_batch = next(iter(dataloader))
print("Images:", images_batch.shape)
print("Labels:", labels_batch.shape)

Images: torch.Size([32, 64, 64])
Labels: torch.Size([32])


In [67]:
from torch.utils.data import random_split

In [68]:
print("Total images:", len(dataset))

Total images: 6000


In [69]:
# 80/20 split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

print("Training images:", train_size)
print("Validation images:", val_size)

Training images: 4800
Validation images: 1200


In [70]:
train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)
print("Training set:", len(train_dataset))
print("Validation set:", len(val_dataset))

Training set: 4800
Validation set: 1200


In [71]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [72]:
images_batch, labels_batch = next(iter(train_loader))

print("Training images:", images_batch.shape)
print("Training labels:", labels_batch.shape)
images_batch, labels_batch = next(iter(val_loader))

print("Validation images:", images_batch.shape)
print("Validation labels:", labels_batch.shape)

print(len(train_dataset) + len(val_dataset))
print(len(dataset))

Training images: torch.Size([32, 64, 64])
Training labels: torch.Size([32])
Validation images: torch.Size([32, 64, 64])
Validation labels: torch.Size([32])
6000
6000
